In [1]:
import numpy as np
import pandas as pd

In [2]:
rng = np.random.default_rng(seed=42)
N_PATIENTS = 2000

In [3]:
print("Pronto! numpy", np.__version__, "| pandas", pd.__version__)

Pronto! numpy 2.5.4 | pandas 3.1.1


In [4]:
CLASS_PROPORTIONS = {
    "nociceptive": 0.40,
    "neuropathic": 0.15,
    "nociplastic": 0.20,
    "mixed": 0.25,
}

labels = rng.choice(
    list(CLASS_PROPORTIONS.keys()),
    size=N_PATIENTS,
    p=list(CLASS_PROPORTIONS.values()),
)

df = pd.DataFrame({
    "patient_id": range(1, N_PATIENTS + 1),
    "mechanism": labels,
})

df["mechanism"].value_counts(normalize=True).round(3)

mechanism
nociceptive    0.394
mixed          0.262
nociplastic    0.191
neuropathic    0.152
Name: proportion, dtype: float64

In [5]:
DURATION_PARAMS = {
    "nociceptive": (2, 0.9),
    "neuropathic": (6, 0.8),
    "nociplastic": (24, 0.7),
    "mixed": (12, 0.8),
}

duration = np.empty(N_PATIENTS)

for mechanism, (median, spread) in DURATION_PARAMS.items():
    mask = df["mechanism"] == mechanism
    duration[mask] = rng.lognormal(mean=np.log(median), sigma=spread, size=mask.sum())

df["pain_duration_months"] = np.clip(duration, 0.25, 240).round(1)

is_nociplastic = df["mechanism"] == "nociplastic"
df.loc[is_nociplastic, "pain_duration_months"] = df.loc[is_nociplastic, "pain_duration_months"].clip(lower=3)

df.groupby("mechanism")["pain_duration_months"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
mechanism,,,,,,,,
mixed,524.0,16.9,15.9,1.5,6.9,12.1,20.4,116.9
neuropathic,305.0,8.8,8.4,0.7,3.6,6.1,11.1,69.4
nociceptive,789.0,3.0,3.3,0.2,1.1,2.1,3.6,26.2
nociplastic,382.0,30.1,25.0,3.5,15.0,24.1,37.2,208.5


In [6]:
DISTRIBUTION_PROBS = {
    "nociceptive": [0.75, 0.05, 0.18, 0.02],
    "neuropathic": [0.10, 0.70, 0.15, 0.05],
    "nociplastic": [0.05, 0.05, 0.35, 0.55],
    "mixed":       [0.25, 0.30, 0.30, 0.15],
}
DISTRIBUTION_CATEGORIES = ["localized", "dermatomal", "regional", "widespread"]

distribution = np.empty(N_PATIENTS, dtype=object)

for mechanism, probs in DISTRIBUTION_PROBS.items():
    mask = df["mechanism"] == mechanism
    distribution[mask] = rng.choice(DISTRIBUTION_CATEGORIES, size=mask.sum(), p=probs)

df["pain_distribution"] = distribution

pd.crosstab(df["mechanism"], df["pain_distribution"], normalize="index").round(2)

pain_distribution,dermatomal,localized,regional,widespread
mechanism,,,,
mixed,0.27,0.25,0.33,0.15
neuropathic,0.78,0.06,0.14,0.03
nociceptive,0.05,0.77,0.16,0.02
nociplastic,0.05,0.06,0.41,0.49


In [7]:
# Questionnaire scores per mechanism: (mean, standard deviation)
DN4_PARAMS = {
    "nociceptive": (1.5, 1.2),
    "neuropathic": (5.5, 1.8),
    "nociplastic": (3.0, 1.5),
    "mixed":       (4.0, 1.8),
}
PAINDETECT_PARAMS = {
    "nociceptive": (8, 4),
    "neuropathic": (22, 5),
    "nociplastic": (15, 5),
    "mixed":       (16, 6),
}

# Empty arrays for the scores
dn4 = np.empty(N_PATIENTS)
paindetect = np.empty(N_PATIENTS)

# Sample scores for each mechanism
for mechanism in CLASS_PROPORTIONS:
    mask = df["mechanism"] == mechanism
    n = mask.sum()

    mean, sd = DN4_PARAMS[mechanism]
    dn4[mask] = rng.normal(loc=mean, scale=sd, size=n)

    mean, sd = PAINDETECT_PARAMS[mechanism]
    paindetect[mask] = rng.normal(loc=mean, scale=sd, size=n)

# Keep scores within each scale and as whole numbers
df["dn4_score"] = np.clip(dn4, 0, 10).round().astype(int)
df["paindetect_score"] = np.clip(paindetect, -1, 38).round().astype(int)

# Check: mean score and % above the cut-off
df.groupby("mechanism").agg(
    dn4_mean=("dn4_score", "mean"),
    dn4_positive=("dn4_score", lambda s: (s >= 4).mean()),
    paindetect_mean=("paindetect_score", "mean"),
    paindetect_positive=("paindetect_score", lambda s: (s >= 19).mean()),
).round(2)

,dn4_mean,dn4_positive,paindetect_mean,paindetect_positive
mechanism,,,,
mixed,3.89,0.56,15.79,0.34
neuropathic,5.29,0.85,22.11,0.74
nociceptive,1.56,0.06,8.03,0.00
nociplastic,3.03,0.38,14.96,0.24


In [8]:
# CSI per mechanism: (mean, standard deviation)
CSI_PARAMS = {
    "nociceptive": (25, 10),
    "neuropathic": (32, 11),
    "nociplastic": (55, 12),
    "mixed":       (42, 13),
}

# Sample CSI for each mechanism
csi = np.empty(N_PATIENTS)

for mechanism in CLASS_PROPORTIONS:
    mask = df["mechanism"] == mechanism
    mean, sd = CSI_PARAMS[mechanism]
    csi[mask] = rng.normal(loc=mean, scale=sd, size=mask.sum())

df["csi_score"] = np.clip(csi, 0, 100).round().astype(int)

# Sleep quality (0-10, 10 = best): worse when CSI is higher
sleep_noise = rng.normal(loc=0, scale=1.5, size=N_PATIENTS)
sleep = 8 - 0.07 * df["csi_score"] + sleep_noise

df["sleep_quality"] = np.clip(sleep, 0, 10).round().astype(int)

# Check: summary by mechanism
df.groupby("mechanism").agg(
    csi_mean=("csi_score", "mean"),
    csi_positive=("csi_score", lambda s: (s >= 40).mean()),
    sleep_mean=("sleep_quality", "mean"),
).round(2)

,csi_mean,csi_positive,sleep_mean
mechanism,,,
mixed,42.16,0.58,5.06
neuropathic,31.92,0.25,5.84
nociceptive,24.92,0.06,6.18
nociplastic,54.30,0.89,4.16


In [9]:
# Check: correlation between CSI and sleep
df[["csi_score", "sleep_quality"]].corr().round(2)

,csi_score,sleep_quality
csi_score,1.00,-0.59
sleep_quality,-0.59,1.00


In [10]:
# Imaging per mechanism:
# (P imaging performed, P concordant tissue finding, P concordant nerve compression)
IMAGING_PARAMS = {
    "nociceptive": (0.35, 0.60, 0.05),
    "neuropathic": (0.70, 0.30, 0.65),
    "nociplastic": (0.60, 0.15, 0.05),
    "mixed":       (0.60, 0.45, 0.35),
}

# Empty True/False arrays
imaging_done = np.empty(N_PATIENTS, dtype=bool)
tissue_finding = np.empty(N_PATIENTS, dtype=bool)
nerve_compression = np.empty(N_PATIENTS, dtype=bool)

# Sample imaging results for each mechanism
for mechanism in CLASS_PROPORTIONS:
    mask = df["mechanism"] == mechanism
    n = mask.sum()
    p_done, p_tissue, p_nerve = IMAGING_PARAMS[mechanism]

    imaging_done[mask] = rng.random(n) < p_done
    tissue_finding[mask] = rng.random(n) < p_tissue
    nerve_compression[mask] = rng.random(n) < p_nerve

# Convert to text: "not_performed", "yes" or "no"
df["imaging_tissue_finding"] = np.where(
    ~imaging_done, "not_performed", np.where(tissue_finding, "yes", "no")
)
df["imaging_nerve_compression"] = np.where(
    ~imaging_done, "not_performed", np.where(nerve_compression, "yes", "no")
)

# Check: % of each answer by mechanism
pd.crosstab(df["mechanism"], df["imaging_tissue_finding"], normalize="index").round(2)

imaging_tissue_finding,no,not_performed,yes
mechanism,,,
mixed,0.34,0.40,0.26
neuropathic,0.50,0.28,0.22
nociceptive,0.14,0.66,0.20
nociplastic,0.53,0.39,0.08


In [11]:
# Latent psychological distress: mean per mechanism (small differences)
DISTRESS_MEANS = {
    "nociceptive": -0.2,
    "neuropathic":  0.0,
    "nociplastic":  0.5,
    "mixed":        0.2,
}

distress = np.empty(N_PATIENTS)
for mechanism in CLASS_PROPORTIONS:
    mask = df["mechanism"] == mechanism
    distress[mask] = rng.normal(loc=DISTRESS_MEANS[mechanism], scale=1, size=mask.sum())


# Helper: build a questionnaire score from the latent distress
def make_score(base, weight, noise_sd, low, high):
    noise = rng.normal(loc=0, scale=noise_sd, size=N_PATIENTS)
    score = base + weight * distress + noise
    return np.clip(score, low, high).round().astype(int)


# Psychosocial scores (base, weight, noise, min, max)
df["hads_anxiety"] = make_score(7, 3, 2, 0, 21)
df["hads_depression"] = make_score(6, 3, 2, 0, 21)
df["pcs_score"] = make_score(18, 8, 6, 0, 52)
df["tsk_score"] = make_score(38, 5, 5, 17, 68)
df["work_stress"] = make_score(5, 1.5, 1.5, 0, 10)

# Check: mean scores by mechanism
psychosocial_cols = ["hads_anxiety", "hads_depression", "pcs_score", "tsk_score", "work_stress"]
df.groupby("mechanism")[psychosocial_cols].mean().round(1)

,hads_anxiety,hads_depression,pcs_score,tsk_score,work_stress
mechanism,,,,,
mixed,7.8,6.7,20.1,38.8,5.4
neuropathic,6.6,6.0,17.8,37.4,4.9
nociceptive,6.8,5.8,17.4,37.5,4.8
nociplastic,8.6,7.5,22.4,40.1,5.7


In [12]:
# Check: correlations between psychosocial scores
df[psychosocial_cols].corr().round(2)

,hads_anxiety,hads_depression,pcs_score,tsk_score,work_stress
hads_anxiety,1.00,0.70,0.68,0.61,0.60
hads_depression,0.70,1.00,0.66,0.59,0.58
pcs_score,0.68,0.66,1.00,0.56,0.56
tsk_score,0.61,0.59,0.56,1.00,0.50
work_stress,0.60,0.58,0.56,0.50,1.00


In [13]:
# Red flags: probability for each one (independent of mechanism)
RED_FLAG_RATES = {
    "rf_cancer_history": 0.03,
    "rf_systemic_signs": 0.02,
    "rf_cauda_equina":   0.005,
    "rf_inflammatory":   0.03,
}

for flag, rate in RED_FLAG_RATES.items():
    df[flag] = rng.random(N_PATIENTS) < rate

# Any red flag present?
red_flag_cols = list(RED_FLAG_RATES.keys())
df["any_red_flag"] = df[red_flag_cols].any(axis=1)

# Check: % of each red flag and overall
df[red_flag_cols + ["any_red_flag"]].mean().round(3)

rf_cancer_history    0.032
rf_systemic_signs    0.014
rf_cauda_equina      0.005
rf_inflammatory      0.027
any_red_flag         0.076
dtype: float64

In [14]:
# Questionnaire columns that can have missing values
questionnaire_cols = [
    "dn4_score", "paindetect_score", "csi_score",
    "hads_anxiety", "hads_depression", "pcs_score", "tsk_score",
]

# Nullable integers: whole numbers that allow missing values
df[questionnaire_cols] = df[questionnaire_cols].astype("Int64")

# 1. MNAR: severe depression scores are left blank more often
severe_depression = df["hads_depression"] >= 11
skip_depression = severe_depression & (rng.random(N_PATIENTS) < 0.25)
df.loc[skip_depression, "hads_depression"] = pd.NA

# 2. MAR: CSI is less often applied in acute pain (< 3 months)
acute_pain = df["pain_duration_months"] < 3
skip_csi = acute_pain & (rng.random(N_PATIENTS) < 0.30)
df.loc[skip_csi, "csi_score"] = pd.NA

# 3. MCAR: random missing values (e.g., lost forms), 4% per questionnaire
for col in questionnaire_cols:
    lost_form = rng.random(N_PATIENTS) < 0.04
    df.loc[lost_form, col] = pd.NA

# Check: % missing per column (only columns with missing values)
missing = df.isna().mean().round(3)
missing[missing > 0]

dn4_score           0.030
paindetect_score    0.041
csi_score           0.136
hads_anxiety        0.042
hads_depression     0.080
pcs_score           0.044
tsk_score           0.042
dtype: float64

In [15]:
# Save the synthetic dataset to the data folder
OUTPUT_PATH = "../data/synthetic_pain_patients.csv"
df.to_csv(OUTPUT_PATH, index=False)

print("Saved", df.shape[0], "patients and", df.shape[1], "columns to", OUTPUT_PATH)

Saved 2000 patients and 20 columns to ../data/synthetic_pain_patients.csv
